<a href="https://colab.research.google.com/github/AnisaIntania/Natural-Language-Processing/blob/main/NLP_P5_04_NER_using_spaCy_CoNLL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import os

In [5]:
# upload train.txt, test.txt, valid.txt from Data/conll2003/en
try:
    from google.colab import files
    uploaded = files.upload()
except ModuleNotFoundError:
    print('Not using colab')

Saving test.txt to test.txt
Saving train.txt to train.txt
Saving valid.txt to valid.txt


In [6]:
#Read the CONLL data from conll2003 folder, and store the formatted data into a folder spacyNER_data

# !mkdir spacyNER_data
os.mkdir('spacyNER_data')

#the above lines create folder if it doesn't exist. If it does, the output shows a message that it
#already exists and cannot be created again
try:
    import google.colab
    !python -m spacy convert "train.txt" spacyNER_data -c ner
    !python -m spacy convert "test.txt" spacyNER_data -c ner
    !python -m spacy convert "valid.txt" spacyNER_data -c ner
except ModuleNotFoundError:
    !python -m spacy convert "Data/conll2003/en/train.txt" spacyNER_data -c ner
    !python -m spacy convert "Data/conll2003/en/test.txt" spacyNER_data -c ner
    !python -m spacy convert "Data/conll2003/en/valid.txt" spacyNER_data -c ner

ℹ Auto-detected token-per-line NER format
ℹ Grouping every 1 sentences into a document.
⚠ To generate better training data, you may want to group sentences
into documents with `-n 10`.
✔ Generated output file (14987 documents):
spacyNER_data/train.spacy
ℹ Auto-detected token-per-line NER format
ℹ Grouping every 1 sentences into a document.
⚠ To generate better training data, you may want to group sentences
into documents with `-n 10`.
✔ Generated output file (3684 documents): spacyNER_data/test.spacy
ℹ Auto-detected token-per-line NER format
ℹ Grouping every 1 sentences into a document.
⚠ To generate better training data, you may want to group sentences
into documents with `-n 10`.
✔ Generated output file (3466 documents): spacyNER_data/valid.spacy


In [7]:
try:
    import google.colab
    !echo "BEFORE : (train.txt)"
    !head "train.txt" -n 11 | tail -n 9
except ModuleNotFoundError:
    print("BEFORE : (Data/conll2003/en/train.txt)")
    file = open("Data/conll2003/en/train.txt")
    content = file.readlines()
    print(*content[1:11])

BEFORE : (train.txt)
EU NNP B-NP B-ORG
rejects VBZ B-VP O
German JJ B-NP B-MISC
call NN I-NP O
to TO B-VP O
boycott VB I-VP O
British JJ B-NP B-MISC
lamb NN I-NP O
. . O O


In [14]:
!ls spacyNER_data

test.spacy  train.spacy  valid.spacy


In [13]:
import spacy
from spacy.tokens import DocBin

nlp = spacy.blank("en")
doc_bin = DocBin().from_disk("spacyNER_data/train.spacy")
docs = list(doc_bin.get_docs(nlp.vocab))

print("AFTER : (spacyNER_data/train.spacy)")

for doc in docs[1:3]:
    print("Teks:", doc.text)
    print("Entitas:", [(ent.text, ent.label_) for ent in doc.ents])
    print()

AFTER : (spacyNER_data/train.spacy)
Teks: EU rejects German call to boycott British lamb . 
Entitas: [('EU', 'ORG'), ('German', 'MISC'), ('British', 'MISC')]

Teks: Peter Blackburn 
Entitas: [('Peter Blackburn', 'PER')]



In [16]:
# Buat config NER tanpa pretrained vectors
!python -m spacy init config config.cfg --lang en --pipeline ner --optimize efficiency --force

# Training model NER
!python -m spacy train config.cfg \
    --output ./model \
    --paths.train ./spacyNER_data/train.spacy \
    --paths.dev ./spacyNER_data/valid.spacy \
    --gpu-id -1

⚠ To generate a more effective transformer-based config (GPU-only),
install the spacy-transformers package and re-run this command. The config
generated now does not use transformers.
ℹ Generated config template specific for your use case
- Language: en
- Pipeline: ner
- Optimize for: efficiency
- Hardware: CPU
- Transformer: None
✔ Auto-filled config with all values
✔ Saved config
config.cfg
You can now add your data and train your pipeline:
python -m spacy train config.cfg --paths.train ./train.spacy --paths.dev ./dev.spacy
ℹ Saving to output directory: model
ℹ Using CPU

=========================== Initializing pipeline ===========================
✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['tok2vec', 'ner']
ℹ Initial learn rate: 0.001
E    #       LOSS TOK2VEC  LOSS NER  ENTS_F  ENTS_P  ENTS_R  SCORE 
---  ------  ------------  --------  ------  ------  ------  ------
  0       0          0.00     50.06    0.97 

In [18]:
import os
os.makedirs("result", exist_ok=True)
!python -m spacy evaluate ./model/model-best ./spacyNER_data/test.spacy

ℹ Using CPU

================================== Results ==================================

TOK     -    
NER P   82.21
NER R   81.43
NER F   81.82
SPEED   13490


=============================== NER (per type) ===============================

           P       R       F
PER    84.38   81.82   83.08
LOC    85.91   88.07   86.97
ORG    78.09   76.82   77.45
MISC   77.86   75.64   76.73



In [19]:
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 85.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [22]:
import spacy
from spacy.tokens import DocBin
from spacy.scorer import Scorer
from spacy.training import Example

# Load model pretrained dan dataset test
nlp = spacy.load("en_core_web_sm")
data_nlp = spacy.blank("en")

doc_bin = DocBin().from_disk("spacyNER_data/test.spacy")
docs = list(doc_bin.get_docs(data_nlp.vocab))

# Samakan label pretrained dengan label CoNLL
label_map = {
    "PERSON": "PER",
    "GPE": "LOC",
    "FAC": "LOC",
    "ORG": "ORG",
    "LOC": "LOC",
    "NORP": "MISC",
    "EVENT": "MISC",
    "PRODUCT": "MISC",
    "WORK_OF_ART": "MISC",
    "LANGUAGE": "MISC"
}

examples = []

for gold_doc in docs:
    # Lewati dokumen penanda
    if gold_doc.text.strip() == "-DOCSTART-":
        continue

    pred_doc = nlp(gold_doc.text)

    # Ubah label prediksi ke skema CoNLL
    new_ents = []
    for ent in pred_doc.ents:
        new_label = label_map.get(ent.label_)
        if new_label:
            new_ents.append(
                (ent.start_char, ent.end_char, new_label)
            )

    # Buat dokumen prediksi dengan label yang sudah disamakan
    pred_doc.ents = [
        pred_doc.char_span(start, end, label=label, alignment_mode="contract")
        for start, end, label in new_ents
        if pred_doc.char_span(
            start, end, label=label, alignment_mode="contract"
        ) is not None
    ]

    examples.append(Example(pred_doc, gold_doc))

# Evaluasi
scorer = Scorer()
scores = scorer.score(examples)

print("Pretrained Model - Label Disamakan")
print("NER Precision:", round(scores["ents_p"], 2))
print("NER Recall:", round(scores["ents_r"], 2))
print("NER F1-score:", round(scores["ents_f"], 2))
print("Per label:", scores["ents_per_type"])

Pretrained Model - Label Disamakan
NER Precision: 0.67
NER Recall: 0.56
NER F1-score: 0.61
Per label: {'ORG': {'p': 0.44869565217391305, 'r': 0.3106562311860325, 'f': 0.36712913553895415}, 'PER': {'p': 0.7353595255744997, 'r': 0.6134817563388992, 'f': 0.6689143627781524}, 'LOC': {'p': 0.7710163111668757, 'r': 0.736810551558753, 'f': 0.7535254445125689}, 'MISC': {'p': 0.7185840707964601, 'r': 0.5783475783475783, 'f': 0.6408839779005525}}
